# Jackett Heroku Deployer

This automated Google Colab notebook will guide you through deploying Jackett to Heroku using its Docker container.

**Instructions:**
Run each cell sequentially. Provide the requested information when prompted.

## Step 1: Install Dependencies
We'll install `jq`, `curl`, `git`, and the Heroku CLI.

In [ ]:
!apt-get update > /dev/null
!apt-get install -y jq curl git > /dev/null
!curl -sL https://cli-assets.heroku.com/install-ubuntu.sh | sh
print('Dependencies installed successfully.')

## Step 2: Collect User Input
Please provide your configuration variables.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import os

api_key_input = widgets.Password(description='Heroku API Key:')
team_name_input = widgets.Text(description='Team Name (Optional):', placeholder='Leave blank for personal')
app_name_input = widgets.Text(description='App Name:', placeholder='my-jackett-app')
region_input = widgets.Dropdown(options=['us', 'eu'], value='us', description='Region:')
repo_url_input = widgets.Text(description='GitHub Repo URL:', value='https://github.com/Jackett/Jackett.git')
branch_input = widgets.Text(description='Branch:', value='master')
git_token_input = widgets.Password(description='GitHub PAT (Optional):', placeholder='For private repos')

display(api_key_input, team_name_input, app_name_input, region_input, repo_url_input, branch_input, git_token_input)
print('Fill out the form above, then proceed to the next cell.')

## Step 3: Authenticate with Heroku
We'll use your API key to authenticate the Heroku CLI.

In [ ]:
api_key = api_key_input.value.strip()
if not api_key:
    raise ValueError('API Key is required.')

os.environ['HEROKU_API_KEY'] = api_key

auth_result = !heroku auth:whoami
if 'Error' in ''.join(auth_result) or 'not logged in' in ''.join(auth_result):
    raise Exception('Authentication failed. Check your API key.')
else:
    print(f"Successfully authenticated as: {auth_result[0]}")

## Step 4: Clone the GitHub Repository
We'll clone the selected repository and checkout the correct branch.

In [ ]:
repo_url = repo_url_input.value.strip()
branch = branch_input.value.strip()
git_token = git_token_input.value.strip()

if git_token:
    repo_url = repo_url.replace('https://', f'https://{git_token}@')

!rm -rf jackett_src
!git clone {repo_url} jackett_src
%cd jackett_src
!git checkout {branch}
print('Repository cloned and branch checked out successfully.')

## Step 5: Create the Heroku Application
We'll create the Heroku application in your specified region, associated with a team if provided.

In [ ]:
app_name = app_name_input.value.strip()
team_name = team_name_input.value.strip()
region = region_input.value

cmd = f"heroku apps:create {app_name} --region {region}"
if team_name:
    cmd += f" --team {team_name}"

print('Creating app...')
!{cmd}
print('App created (or already exists).')

## Step 6: Configure Environment Variables
We'll automatically apply the required environment variables.

In [ ]:
print('Setting config vars...')
!heroku config:set HEROKU=true ASPNETCORE_ENVIRONMENT=Production -a {app_name}
print('Environment variables configured.')

## Step 7: Build the Docker Image
Building the Jackett multi-stage docker image...

In [ ]:
print('Building Docker Image. This will take a few minutes...')
build_res = !docker build -t registry.heroku.com/{app_name}/web .
if any('error' in str(line).lower() for line in build_res[-10:]):
    print('\n'.join(build_res))
    raise Exception('Docker build failed.')
else:
    print('Docker build succeeded.')

## Step 8: Login to Heroku Container Registry
We'll authenticate Docker with the Heroku registry.

In [ ]:
!heroku container:login
print('Logged into Heroku Container Registry.')

## Step 9: Push the Container
Pushing the built image to Heroku.

In [ ]:
print('Pushing image... Please wait.')
push_res = !docker push registry.heroku.com/{app_name}/web
if any('denied' in str(line).lower() for line in push_res):
    print('\n'.join(push_res))
    raise Exception('Push failed.')
else:
    print('Push succeeded.')

## Step 10: Release the Container
We will release the container to the web dyno.

In [ ]:
!heroku container:release web -a {app_name}
print('Container released successfully.')

## Step 11: Deployment Verification
We will wait a few seconds and then test the `/health` endpoint.

In [ ]:
import time
import requests

url = f"https://{app_name}.herokuapp.com"
health_url = f"{url}/health"

print('Waiting for application to boot (15 seconds)...')
time.sleep(15)

try:
    response = requests.get(health_url, timeout=10)
    if response.status_code == 200 and 'OK' in response.text:
        print('\n✅ Deployment Successful!')
        print(f'App URL: {url}')
        print(f'Health Endpoint: {health_url} returned OK')
        print(f'Dashboard URL: https://dashboard.heroku.com/apps/{app_name}')
    else:
        print('\n⚠️ Deployment might have failed or app is still booting.')
        print(f'Status Code: {response.status_code}, Response: {response.text}')
        print('\n--- Recent Logs ---')
        !heroku logs --tail -n 20 -a {app_name}
except Exception as e:
    print('\n❌ Failed to connect to health endpoint.')
    print(str(e))
    print('\n--- Recent Logs ---')
    !heroku logs --tail -n 20 -a {app_name}

## Step 12: Application Management (Optional)
Use the cells below to manage your deployed application.

In [ ]:
# Restart Dynos
# !heroku ps:restart -a {app_name}

In [ ]:
# View Config Vars
# !heroku config -a {app_name}

In [ ]:
# View Live Logs
# !heroku logs --tail -a {app_name}

In [ ]:
# Delete Application
# Uncomment the line below to completely destroy the application
# !heroku apps:destroy -a {app_name} --confirm {app_name}